# UrbanPulse — Notebook 06: Traffic Intelligence Engine

Bible §6. A **rule-based** reasoning layer (not a model): Road Health Score + metabolic state, Congestion Risk Score, hotspot ranking, critical flags, severity alerts, and **archetype-aware recommendations with reasoning**.

Archetype inputs (ECHO B7) and cascade flags (ECHO B8) are optional; archetype rules stay silent until B7 assigns archetypes. **Produces:** `reports/engine/snapshot_*.json`, `reports/engine/rule_activation_census.csv`.

In [1]:
import sys, pathlib, joblib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import config, io_utils
from engine import intelligence
import pandas as pd
model = joblib.load(config.BEST_MODEL_PKL)
features = io_utils.load_parquet(config.FEATURES_PARQUET)

## 1. Rule-activation census — which §6.3 rules are live on this data?
Note R5 (Landmine) never fires: `lane5_stalled` and a >400s queue never co-occur. R3 (Saturator) fires mostly on Link 36 — a hint that B7 will likely re-classify Link 36 as a Saturator, not the Bible's placeholder 'Landmine'.

In [2]:
intelligence.rule_activation_census(features)

,rule,condition,fires
0,R2 (Any),speed_div>5 & queue>300,182
1,R3 (Saturator),max_occup>=0.9 & queue>500,121
2,R4 (Chronic),health<30 x3 consecutive,5
3,R5 (Landmine),lane5_stalled & max_queue>400,0
4,R6 (Any),queue surge>200s,26


## 2. Snapshot — Day 4, 08:10 (a queue-surge incident)

In [3]:
res = intelligence.analyze_snapshot(features, model, day_number=4, minute_of_day=490)
print('links:', res['n_links'], '| critical:', res['n_critical'])

links: 66 | critical: 16


## 3. Hotspot ranking (worst 10)

In [4]:
pd.DataFrame(res['hotspot_ranking']).head(10)

,link_id,health_score,state
0,17,38.63,Saturated
1,44,43.19,Stressed
2,5,43.45,Stressed
3,10,43.92,Stressed
4,11,44.36,Stressed
5,13,46.32,Stressed
6,30,46.91,Stressed
7,31,47.31,Stressed
8,46,47.32,Stressed
9,6,48.11,Stressed


## 4. Recommendations (each with trigger + reasoning)

In [5]:
for link in res['links']:
    for rec in link['recommendations']:
        print(f"[Link {link['link_id']}] ({rec['severity']}) {rec['recommendation']}")
        print(f"   why: {rec['reasoning']}")

[Link 17] (ADVISORY) Issue ADVISORY to downstream links
   why: Queue jumped 211s in one interval — likely an incident; downstream links should prepare for spillback.
[Link 23] (ADVISORY) Issue ADVISORY to downstream links
   why: Queue jumped 278s in one interval — likely an incident; downstream links should prepare for spillback.
[Link 27] (ADVISORY) Issue ADVISORY to downstream links
   why: Queue jumped 249s in one interval — likely an incident; downstream links should prepare for spillback.
[Link 29] (ADVISORY) Issue ADVISORY to downstream links
   why: Queue jumped 245s in one interval — likely an incident; downstream links should prepare for spillback.


## 5. Critical roads + alerts (worst 5)

In [6]:
for link in res['links'][:0] or sorted(res['links'], key=lambda l: l['health_score'])[:5]:
    print(f"Link {link['link_id']}: {link['state']} health={link['health_score']} P15={link['congestion_prob']}")
    for a in link['alerts']:
        print('   ', a['severity'], '-', a['message'])

Link 17: Saturated health=38.63 P15=0.9876
    WARNING - Road is Saturated (warning).
    CRITICAL - Critical flag: P(congested in 15 min)=0.99, queue=502s.
Link 44: Stressed health=43.19 P15=0.9949
    ADVISORY - Road is Stressed (advisory).
    CRITICAL - Critical flag: P(congested in 15 min)=0.99, queue=299s.
Link 5: Stressed health=43.45 P15=0.9999
    ADVISORY - Road is Stressed (advisory).
    CRITICAL - Critical flag: P(congested in 15 min)=1.00, queue=419s.
Link 10: Stressed health=43.92 P15=0.9978
    ADVISORY - Road is Stressed (advisory).
    CRITICAL - Critical flag: P(congested in 15 min)=1.00, queue=318s.
Link 11: Stressed health=44.36 P15=0.0049
    ADVISORY - Road is Stressed (advisory).


## 6. Gate — every recommendation carries reasoning

In [7]:
ok = all(r['reasoning'].strip() for l in res['links'] for r in l['recommendations'])
print('GATE:', 'PASS' if ok else 'FAIL')

GATE: PASS
